In [ ]:
from xgolib import XGO
from key import Button
import ipywidgets.widgets as widgets
import time
import socket
import json
import sys
import xgoscreen.LCD_2inch as LCD_2inch
from PIL import Image,ImageDraw,ImageFont
from IPython.display import display 


In [ ]:
_font_cache = {}
def get_font(size):
    if size not in _font_cache:
        _font_cache[size] = ImageFont.truetype("/home/pi/model/msyh.ttc", size)
    return _font_cache[size]

#屏幕清除
mydisplay = LCD_2inch.LCD_2inch()
mydisplay.clear()
font1 = get_font(30)
splash = Image.new("RGB", (mydisplay.height, mydisplay.width ),"black")
draw = ImageDraw.Draw(splash)
draw.rectangle((0, 0, mydisplay.height, mydisplay.width), fill="BLACK")
draw.text((50, 80), "Teaching Class!!!", fill="WHITE", font=font1)
mydisplay.ShowImage(splash)




In [ ]:
# 初始化
dog = XGO(port='/dev/ttyAMA0', version="xgomini")
button = Button()


# 网络设置
UDP_IP = "255.255.255.255"  # 广播地址
UDP_PORT = 5005
UDP_TIMEOUT = 0.1  # 缩短超时时间，提高响应速度

In [ ]:
#主机发送 Host sends
def send_arm_angles(angles):
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_BROADCAST, 1)
        sock.settimeout(UDP_TIMEOUT)
        data = json.dumps({"arm_angles": angles}).encode('utf-8')
        sock.sendto(data, (UDP_IP, UDP_PORT))
        print(f"[Host] Sent angles: {angles}")
    except Exception as e:
        print(f"[Host] Send failed: {e}")
    finally:
        if 'sock' in locals():
            sock.close()

#从机接收     Receive from the machine        
def receive_arm_angles():
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        sock.settimeout(UDP_TIMEOUT)  # 使用设定的超时时间 Use the set timeout period
        sock.bind(('0.0.0.0', UDP_PORT))  # 明确绑定所有接口 Clearly bind all interfaces
        
        data, addr = sock.recvfrom(1024)
        print(f"[slave] Received raw data: {data}")  # 调试打印 Debugging printing
        try:
            data = json.loads(data.decode('utf-8'))
            if "arm_angles" in data:
                print(f"[slave] Success: {data['arm_angles']} (comr from {addr[0]})")
                return data["arm_angles"], "success"
            else:
                print("[slave] Missing in data arm_angles")
                return None, "invalid_data"
        except json.JSONDecodeError as e:
            print(f"[slave] JSON error: {e}")
            return None, "json_error"
    except socket.timeout:
        print("[slave] Receiving timeout (normal phenomenon)")
        return None, "timeout"
    except Exception as e:
        print(f"[slave] Abnormal reception: {e}")
        return None, "error"
    finally:
        if 'sock' in locals():
            sock.close()


In [ ]:
def check_exit_buttons():
    if button.press_b():
        #添加舵机加载力矩 Add servo motor loading torque
        dog.load_motor(5)
        time.sleep(.02)
        dog.reset()
        print("mode_exit!")
        #sys.exit(0)
        return True
    return False

def host_mode():
    """主机模式  Host mode"""
    time.sleep(1)
    dog.unload_motor(5)
    while True:
        # 显示界面 display interface
        draw.rectangle((0, 0, mydisplay.height, mydisplay.width), fill="BLACK")
        draw.text((20, 80), "Host Mode Running", fill="CYAN", font=font1)
        draw.text((20, 120), "B:Force Exit", fill="WHITE", font=font1)
        mydisplay.ShowImage(splash)
        
        # 业务逻辑 business logic
        motor_data = dog.read_motor()
        print(type(motor_data))
        if motor_data:
            send_arm_angles(motor_data[-3:])
        
        # 按钮检测 Button detection
        if check_exit_buttons():
            break
            
        time.sleep(0.1)

def client_mode():
    """从机模式  slave mode"""
    dog.load_allmotor()
    while True:
        # display interface
        draw.rectangle((0, 0, mydisplay.height, mydisplay.width), fill="BLACK")
        draw.text((20, 80), "Client Mode Running", fill="CYAN", font=font1)
        draw.text((20, 120), "B:Force Exit", fill="WHITE", font=font1)
        mydisplay.ShowImage(splash)
        
        # Business logic (non blocking)
        angles, status = receive_arm_angles()
        if status == "success":
            dog.motor([51, 52, 53], angles)
        
        if check_exit_buttons():
            break
            
        time.sleep(0.1)

In [ ]:
hostbutton = widgets.Button(description="host mode")
clientbutton = widgets.Button(description="client mode")
display(hostbutton)
display(clientbutton)

def on_button_clicked_host(b):
    dog.attitude('p', 15)
    dog.translation('z', 75)
    hostbutton.button_style = 'success'
    clientbutton.button_style = ''
    print("hostbutton clicked - activated")
    host_mode()
    draw.rectangle((0, 0, mydisplay.height, mydisplay.width), fill="BLACK")
    draw.text((60, 80), "Teaching Class!!!", fill="WHITE", font=font1)
    mydisplay.ShowImage(splash)
    b.button_style = ''
   

    

def on_button_clicked_slave(b):
    dog.attitude('p', 15)
    dog.translation('z', 75)
    time.sleep(.5)
    clientbutton.button_style = 'info'
    hostbutton.button_style = ''
    print("clientbutton clicked - activated")
    client_mode()
    draw.rectangle((0, 0, mydisplay.height, mydisplay.width), fill="BLACK")
    draw.text((60, 80), "Teaching Class!!!", fill="WHITE", font=font1)
    mydisplay.ShowImage(splash)
    b.button_style = ''
    

hostbutton.on_click(on_button_clicked_host)
clientbutton.on_click(on_button_clicked_slave)
print("EN:Please press the bottom left button on the screen to end the mode ")
print("CN:请按下屏幕的左下按键结束模式")

In [ ]:
# print("If terminated illegally, please run this source code and load the torque of the servo motor")
# print("如果非法终止，请运行下此段源码，加载舵机的扭力")
dog.load_allmotor()